In [10]:
# get api key here: https://www.census.gov/data/developers/data-sets/acs-5year/2024.html
%pip install pandas requests python-dotenv matplotlib geopandas pygris


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import pygris

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")

if not CENSUS_API_KEY:
    raise ValueError("CENSUS_API_KEY is missing from .env")

In [12]:
url = "https://api.census.gov/data/2024/acs/acs5"

params = {
    "get": (
        "NAME,"
        "B19013_001E,"
        "B08201_001E,"
        "B08201_002E,"
        "B08301_001E,"
        "B08301_010E"
    ),
    "for": "tract:*",
    "in": "state:06 county:075",
    "key": CENSUS_API_KEY
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()

acs = pd.DataFrame(data[1:], columns=data[0])

In [13]:
# RENAME COLS
acs = acs.rename(columns={
    "NAME": "tract_name",
    "B19013_001E": "median_household_income",
    "B08201_001E": "total_households",
    "B08201_002E": "households_without_vehicle",
    "B08301_001E": "total_workers",
    "B08301_010E": "public_transit_commuters"
})

In [15]:
# CONVERT TO PROPER DTYPES and CREATE PERCENTAGE COLS
numeric_cols = [
    "median_household_income",
    "total_households",
    "households_without_vehicle",
    "total_workers",
    "public_transit_commuters"
]

acs[numeric_cols] = acs[numeric_cols].apply(
    pd.to_numeric, errors="coerce"
)

acs[numeric_cols] = acs[numeric_cols].where(acs[numeric_cols] >= 0)

In [16]:
#builds new columns for analysis
acs["pct_without_vehicle"] = (
    acs["households_without_vehicle"]
    / acs["total_households"].replace(0, np.nan) * 100
)

acs["pct_transit_commuters"] = (
    acs["public_transit_commuters"]
    / acs["total_workers"].replace(0, np.nan) * 100
)

acs["income_topcoded"] = acs["median_household_income"] >= 250001

#joins GEOID to merge with future tract boundary shapes
acs["GEOID"] = (
    acs["state"] + acs["county"] + acs["tract"]
)


In [17]:
acs.to_csv("../data/processed/sf_census_tracts.csv", index=False)

In [19]:
path = "../data/processed/sf_segments_clean.csv"
print(os.path.exists(path))

seg = pd.read_csv(path, usecols=["from_stop_id", "to_stop_id"], dtype="string")
print(seg.shape)

True
(4820690, 2)


In [20]:
#loads map shapes and Muni stop locations

#downloads outlines of every census tract and attaches census table by matching GEOID
tr = pygris.tracts(state="CA", county="075", year=2024, cb=True).to_crs(32610)
tr = tr[["GEOID", "geometry"]].merge(acs, on="GEOID", how="left")
tr["tract_area"] = tr.geometry.area

#checks how many tracts failed to match
print(len(tr), "tracts;", tr["total_households"].isna().sum(), "with no census row")
print("invalid geometries:", (~tr.geometry.is_valid).sum())

#ensure each stop ID is only printed once
stops = pd.read_csv("../data/march_2025_gtfs/stops.txt", dtype={"stop_id": "string"})
print("duplicate stop_ids:", stops["stop_id"].duplicated().sum())

#keep only stops that are in Muni segments
used = set(seg["from_stop_id"]) | set(seg["to_stop_id"])
stops = stops[stops["stop_id"].isin(used)].copy()
print(len(stops), "of", len(used), "Muni stop IDs found in stops.txt")

#ensure latitude and longitude are numbers, tunring anything else into NaN
stops["stop_lat"] = pd.to_numeric(stops["stop_lat"], errors="coerce")
stops["stop_lon"] = pd.to_numeric(stops["stop_lon"], errors="coerce")
print("missing coordinates:", stops[["stop_lat", "stop_lon"]].isna().any(axis=1).sum())

#see if any stops are outside the rough SF area
outside = ~stops["stop_lat"].between(37.6, 37.9) | ~stops["stop_lon"].between(-123.2, -122.3)
print("stops outside SF bounding box:", outside.sum())
print(stops.loc[outside, ["stop_id", "stop_name", "stop_lat", "stop_lon"]].head(10))

Using FIPS code '06' for input 'CA'
242 tracts; 0 with no census row
invalid geometries: 0
duplicate stop_ids: 0
3234 of 3234 Muni stop IDs found in stops.txt
missing coordinates: 0
stops outside SF bounding box: 0
Empty DataFrame
Columns: [stop_id, stop_name, stop_lat, stop_lon]
Index: []


In [21]:
#check if any census tracts don't have a map shape
missing = acs[~acs["GEOID"].isin(tr["GEOID"])]
print(missing[["tract_name", "total_households", "total_workers"]])

                                            tract_name  total_households  \
242  Census Tract 9901; San Francisco County; Calif...                 0   
243  Census Tract 9902; San Francisco County; Calif...                 0   

     total_workers  
242              0  
243              0  


In [22]:
#turn each stop's latitude and longitude into a point on the map
pts = gpd.GeoDataFrame(
    stops[["stop_id"]],
    geometry=gpd.points_from_xy(stops["stop_lon"], stops["stop_lat"], crs=4326),
).to_crs(32610)

buf = pts.copy()
buf["geometry"] = pts.buffer(400)

# calculate the share of the tract inside the circle for each piece
pieces = gpd.overlay(buf, tr, how="intersection")
pieces["frac"] = pieces.geometry.area / pieces["tract_area"]

#multiply by frac to estimate the number for each category
pieces["hh"]       = pieces["total_households"] * pieces["frac"]
pieces["hh_noveh"] = pieces["households_without_vehicle"] * pieces["frac"]
pieces["workers"]  = pieces["total_workers"] * pieces["frac"]
pieces["transit"]  = pieces["public_transit_commuters"] * pieces["frac"]

#finding the income average, setting no income estimates to 0
pieces["hh_inc"]   = pieces["hh"].where(pieces["median_household_income"].notna(), 0)
pieces["inc_x_hh"] = pieces["median_household_income"] * pieces["hh_inc"]

#group the slices by stop and sum all the columns
g = pieces.groupby("stop_id")[["hh", "hh_noveh", "workers", "transit", "hh_inc", "inc_x_hh"]].sum()

#make the dataframe with 4 columns per sotp
catch = pd.DataFrame({
    "households_in_buffer":  g["hh"],
    "pct_without_vehicle":   g["hh_noveh"] / g["hh"].replace(0, np.nan) * 100,
    "pct_transit_commuters": g["transit"] / g["workers"].replace(0, np.nan) * 100,
    "avg_median_income":     g["inc_x_hh"] / g["hh_inc"].replace(0, np.nan),
}).reset_index()

#making the csv file
catch.to_csv("../data/processed/sf_stop_catchments.csv", index=False)

#sanity check
print(len(catch), "of", len(stops), "stops have a catchment")
print(catch.describe())

3234 of 3234 stops have a catchment
       households_in_buffer  pct_without_vehicle  pct_transit_commuters  \
count           3234.000000          3232.000000            3232.000000   
mean            2121.808220            25.925555              17.997906   
std             1726.741710            17.972753               5.620016   
min                0.000000             1.634473               0.000000   
25%             1061.354086            12.469773              14.072055   
50%             1562.017953            20.815518              17.684750   
75%             2571.885765            33.185724              21.672665   
max            11873.168522            92.173913              37.154816   

       avg_median_income  
count        3226.000000  
mean       148241.374387  
std         48033.481191  
min         12908.000000  
25%        116255.596918  
50%        151837.358142  
75%        183382.701707  
max        249385.507584  


In [23]:
#find any stops that have missing values and display their names
nan_stops = catch[catch[["pct_without_vehicle", "avg_median_income"]].isna().any(axis=1)]
print(nan_stops.merge(stops[["stop_id", "stop_name"]], on="stop_id"))

  stop_id  households_in_buffer  pct_without_vehicle  pct_transit_commuters  \
0   14562             32.001466            19.327731                9.13242   
1   14873              0.000000                  NaN                    NaN   
2   14874              0.000000                  NaN                    NaN   
3   17286             32.001466            19.327731                9.13242   
4   17353             32.001466            19.327731                9.13242   
5   17364             32.001466            19.327731                9.13242   
6   17377             32.001466            19.327731                9.13242   
7   17378             32.001466            19.327731                9.13242   

   avg_median_income                stop_name  
0                NaN     Evans Ave & Selby St  
1                NaN          Legion Of Honor  
2                NaN          Legion Of Honor  
3                NaN     Evans Ave & Selby St  
4                NaN  Third Street & Marin St  


In [25]:
#tracts behind three of the stops with no income value
ids = ["14562", "17353", "17377"]
print(pieces[pieces["stop_id"].isin(ids)][
    ["stop_id", "tract_name", "frac", "total_households", "median_household_income"]
])

      stop_id                                         tract_name     frac  \
5701    14562  Census Tract 9809; San Francisco County; Calif...  0.13446   
14768   17377  Census Tract 9809; San Francisco County; Calif...  0.13446   
14830   17353  Census Tract 9809; San Francisco County; Calif...  0.13446   

       total_households  median_household_income  
5701                238                      NaN  
14768               238                      NaN  
14830               238                      NaN  
